# Large Language Model Serving Tutorial with DigitalHub

This notebook shows how to deploy a pre-trained language model with the DigitalHub SDK. It serves a DistilBERT sentiment classifier as a REST API.

## Overview
- **Model Selection**: Use a pre-trained DistilBERT model from HuggingFace Hub
- **Model Serving**: Deploy the model as a REST API with GPU acceleration
- **Inference**: Test the model with text classification requests
- **Integration**: Use DigitalHub serving infrastructure

## Setup and Model Configuration

Set up the DigitalHub project and configure the HuggingFace model for serving.

## Project Initialization

Initialize a DigitalHub project using the same naming pattern as the other tutorials.

In [ ]:
import os

import digitalhub as dh

p_name = f"tutorial-project-{os.environ['USER']}"
project = dh.get_or_create_project(p_name)

## Step 1: Model Configuration

Create a function that serves the DistilBERT model directly from HuggingFace Hub.

In [ ]:
llm_function = project.new_function(
    name="sentiment-classifier",
    kind="huggingfaceserve",
    model_name="sentiment-model",
    path="huggingface://distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)

## Step 2: Model Serving

Deploy the model as a REST API service with GPU acceleration for faster inference.

In [ ]:
llm_run = llm_function.run("serve", profile="1xV100", wait=True)

Check that the service is running and ready to accept requests:

In [ ]:
service = llm_run.refresh().status.service
print("Service status:", service)

### Test the LLM API

Test the deployed sentiment classifier with sample text.

In [ ]:
# Prepare test data for sentiment classification
model_name = "sentiment-model"
json_payload = {
    "inputs": [
        {
            "name": "input-0",
            "shape": [2],
            "datatype": "BYTES",
            "data": ["Hello, my dog is cute", "I am feeling sad"],
        },
    ]
}

In [ ]:
# Make prediction request to the deployed LLM
result = llm_run.invoke(model_name=model_name, json=json_payload).json()
print("Sentiment classification results:")
print(result)